# Entrega 3 — indexação do manual GoBR

Este notebook lê `dados/govendas-rules.pdf` com **Docling** e cria um índice vetorial **separado** do índice das 246 explicações. Foi preparado para o Google Colab com GPU T4, mas também funciona em CPU.

### Por que dividir assim?

O manual tem 62 páginas, seções numeradas (por exemplo, `4.3.2`), exemplos de sintaxe e tabelas que atravessam páginas. Um corte a cada N caracteres ou por página separaria o exemplo da explicação e poderia retirar o cabeçalho de uma linha de tabela. O Docling primeiro reconhece títulos, parágrafos, listas e tabelas; o `HybridChunker` só divide quando o trecho passa do limite de tokens, e pode reunir partes pequenas sob o **mesmo caminho de títulos**. Cabeçalhos de tabelas são repetidos nos fragmentos da mesma tabela. Não há sobreposição cega: o contexto vem dos títulos, e as páginas físicas são guardadas para citar a fonte.

**Limite inicial: 768 tokens na contagem do Docling.** O tokenizador do modelo acrescenta tokens especiais ao texto (normalmente dois), por isso a verificação do modelo usa um limite separado de 1024 tokens. É um ponto de partida para busca, não um valor universal. O notebook mostra o tamanho dos trechos e uma amostra de seções sensíveis. Se o Docling reconhecer errado um título, tabela ou bloco de código, a amostra e os avisos indicarão onde revisar antes de usar o índice em RAG. Capa e índice impresso (páginas 1–4) não entram na indexação; as páginas 5–62 contêm o corpo do manual.

[Chunking do Docling](https://docling-project.github.io/docling/concepts/chunking/) · [Hierarquia de títulos em PDF](https://docling-project.github.io/docling/usage/heading_levels/) · [Manual do HybridChunker](https://docling-project.github.io/docling/_generated/examples/hybrid_chunking/)

In [1]:
%pip install -q "docling>=2.132,<3" "llama-index-core>=0.13,<0.15" "llama-index-embeddings-huggingface>=0.7,<0.8" "sentence-transformers>=3,<6" "transformers>=4.41,<5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Localizar o PDF

No Colab, o repositório público é clonado quando não está presente. Defina `GOBR_REPO_DIR` se quiser usar um checkout específico. Execute após o `git push` do notebook para que o Colab receba a versão atual.

In [2]:
from pathlib import Path
import os
import subprocess

URL_REPOSITORIO = "https://github.com/buenofgustavo/projeto-sistemas-multiagentes.git"
NOME_REPOSITORIO = "projeto-sistemas-multiagentes"
REL_PDF = Path("dados/govendas-rules.pdf")

def contem_manual(pasta: Path) -> bool:
    return (pasta / REL_PDF).is_file()

pasta_definida = os.environ.get("GOBR_REPO_DIR")
if pasta_definida:
    REPO_DIR = Path(pasta_definida).expanduser().resolve()
    if not contem_manual(REPO_DIR):
        raise FileNotFoundError(f"GOBR_REPO_DIR não contém {REL_PDF}: {REPO_DIR}")
else:
    candidatos = [Path.cwd(), Path.cwd() / NOME_REPOSITORIO,
                  Path("/workspace") / NOME_REPOSITORIO]
    REPO_DIR = next((p.resolve() for p in candidatos if contem_manual(p)), None)
    if REPO_DIR is None:
        destino = Path.cwd() / NOME_REPOSITORIO
        if destino.exists():
            destino = next((Path.cwd() / f"{NOME_REPOSITORIO}_{n}" for n in range(2, 100)
                           if not (Path.cwd() / f"{NOME_REPOSITORIO}_{n}").exists()), None)
            if destino is None:
                raise RuntimeError("Não há pasta livre para clonar o repositório")
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        URL_REPOSITORIO, str(destino)], check=True)
        REPO_DIR = destino.resolve()

PDF = REPO_DIR / REL_PDF
print("Repositório:", REPO_DIR)
print("Manual:", PDF, "|", PDF.stat().st_size, "bytes")

Repositório: /content/projeto-sistemas-multiagentes
Manual: /content/projeto-sistemas-multiagentes/dados/govendas-rules.pdf | 159326 bytes


## Conversão, chunks e cache

A conversão usa a camada textual existente no PDF (`do_ocr=False`), reconhecimento de tabelas e reconstrução da hierarquia dos títulos numerados. O Docling trabalha no documento estruturado, sem cortar na mudança de página. As páginas originais de cada chunk são preservadas em `paginas_pdf`.

O cache é identificado pelo SHA-256 do PDF e pelas versões/configurações usadas. Ele fica em `gobr_index_cache/manual` fora da pasta clonada no Colab. Ao mudar o PDF ou a estratégia de divisão, novos chunks e embeddings serão produzidos.

In [3]:
from __future__ import annotations

import gc
import hashlib
import json
import sys
from importlib.metadata import version
from statistics import median

import torch
from docling.chunking import HybridChunker
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import HeadingHierarchyOptions, PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.transforms.chunker.hierarchical_chunker import ChunkingDocSerializer, ChunkingSerializerProvider
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from docling_core.transforms.serializer.markdown import MarkdownTableSerializer
from transformers import AutoTokenizer

MODELO_EMBEDDING = "BAAI/bge-m3"
MAX_CHUNK_TOKENS = 768
MAX_MODELO_TOKENS = 1024
VERSAO_PIPELINE = "manual-docling-hybrid-v2"
CACHE_BASE = REPO_DIR.parent / "gobr_index_cache" / "manual"

class MDTableSerializerProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc):
        return ChunkingDocSerializer(doc=doc, table_serializer=MarkdownTableSerializer())

def identidade_pipeline(pdf: Path) -> tuple[str, dict]:
    config = {
        "pipeline": VERSAO_PIPELINE,
        "sha256_pdf": hashlib.sha256(pdf.read_bytes()).hexdigest(),
        "modelo_embedding": MODELO_EMBEDDING,
        "max_chunk_tokens": MAX_CHUNK_TOKENS,
        "paginas_pdf": "5-ultima",
        "ocr": False,
        "tabelas": True,
        "hierarquia_titulos": True,
        "versoes": {p: version(p) for p in (
            "docling", "docling-core", "transformers", "llama-index-core",
            "llama-index-embeddings-huggingface", "sentence-transformers",
        )},
    }
    chave = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]
    return chave, config

CHAVE, CONFIG = identidade_pipeline(PDF)
PASTA_CACHE = CACHE_BASE / CHAVE
ARQUIVO_CHUNKS = PASTA_CACHE / "chunks.jsonl"
PASTA_INDICE = PASTA_CACHE / "indice"
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "não disponível")
print("Cache:", PASTA_CACHE)

GPU: Tesla T4
Cache: /content/gobr_index_cache/manual/fd183f3b452fff50a249


In [4]:
def converter_manual(pdf: Path):
    opcoes = PdfPipelineOptions()
    opcoes.do_ocr = False  # o PDF fornecido possui texto selecionável
    opcoes.do_table_structure = True
    opcoes.generate_parsed_pages = True  # auxilia a hierarquia por estilo quando necessário
    opcoes.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
    opcoes.accelerator_options = AcceleratorOptions(
        device=AcceleratorDevice.CUDA if torch.cuda.is_available() else AcceleratorDevice.CPU
    )
    conversor = DocumentConverter(
        format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=opcoes)}
    )
    # Página física 5 inicia o corpo; o fim é aberto para não truncar versões futuras.
    documento = conversor.convert(pdf, page_range=(5, sys.maxsize)).document
    return documento

def extrair_chunks(pdf: Path, arquivo_saida: Path) -> list[dict]:
    tokenizer_hf = AutoTokenizer.from_pretrained(MODELO_EMBEDDING)
    tokenizer = HuggingFaceTokenizer(tokenizer=tokenizer_hf, max_tokens=MAX_CHUNK_TOKENS)
    chunker = HybridChunker(
        tokenizer=tokenizer,
        merge_peers=True,
        repeat_table_header=True,
        serializer_provider=MDTableSerializerProvider(),
    )
    documento = converter_manual(pdf)
    registros = []
    for i, chunk in enumerate(chunker.chunk(dl_doc=documento), 1):
        texto = chunker.contextualize(chunk=chunk).strip()
        if not texto:
            continue
        paginas = sorted({prov.page_no for item in chunk.meta.doc_items
                          for prov in (item.prov or [])})
        if not paginas:
            raise ValueError(f"Chunk {i} sem página de origem; revise a conversão")
        tokens_docling = tokenizer.count_tokens(texto)
        tokens_modelo = len(tokenizer_hf.encode(texto, truncation=False))
        if tokens_docling > MAX_CHUNK_TOKENS:
            raise ValueError(f"Chunk {i} excede o limite do Docling: {tokens_docling} tokens")
        if tokens_modelo > MAX_MODELO_TOKENS:
            raise ValueError(f"Chunk {i} excede o limite do embedding: {tokens_modelo} tokens")
        registros.append({
            "id": f"manual_{i:04d}",
            "texto": texto,
            "conteudo": chunk.text.strip(),
            "titulos": chunk.meta.headings or [],
            "legendas": chunk.meta.captions or [],
            "paginas_pdf": paginas,
            "itens_docling": [item.self_ref for item in chunk.meta.doc_items],
            "tipos_itens": sorted({str(item.label) for item in chunk.meta.doc_items}),
            "tokens": tokens_modelo,
            "tokens_docling": tokens_docling,
        })
    if not registros:
        raise ValueError("O Docling não produziu conteúdo indexável")
    arquivo_saida.parent.mkdir(parents=True, exist_ok=True)
    arquivo_saida.write_text("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in registros),
                             encoding="utf-8")
    del documento, chunker
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return registros

if ARQUIVO_CHUNKS.is_file():
    registros = [json.loads(linha) for linha in ARQUIVO_CHUNKS.read_text(encoding="utf-8").splitlines()
                 if linha.strip()]
    print("Chunks carregados do cache:", ARQUIVO_CHUNKS)
else:
    print("Convertendo o manual com Docling; a primeira execução baixa modelos de layout/tabela.")
    registros = extrair_chunks(PDF, ARQUIVO_CHUNKS)
    print("Chunks criados:", ARQUIVO_CHUNKS)
assert registros, "O arquivo de chunks está vazio"
assert all(r["texto"].strip() and r["paginas_pdf"]
           and r["tokens_docling"] <= MAX_CHUNK_TOKENS
           and r["tokens"] <= MAX_MODELO_TOKENS
           for r in registros), "Cache inválido ou chunk sem origem"
print("Total de chunks:", len(registros))

Convertendo o manual com Docling; a primeira execução baixa modelos de layout/tabela.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/tmp/ipykernel_817/158711480.py:47: DeprecationWarning: deprecated
  "legendas": chunk.meta.captions or [],


Chunks criados: /content/gobr_index_cache/manual/fd183f3b452fff50a249/chunks.jsonl
Total de chunks: 96


## Conferir os cortes antes de indexar

O resumo mostra distribuição de tamanhos e páginas, e sinaliza itens Docling repetidos em mais de um chunk (isso pode ocorrer em tabelas longas). Leia especialmente os exemplos que cruzam páginas e as tabelas. O notebook imprime amostras das seções `4.3.2` (carteiras), `7.4` (frequência) e `8.2` (alerta completo). Se um bloco de código tiver sido cortado de modo ruim ou uma tabela perder coluna, ajuste `MAX_CHUNK_TOKENS` ou corrija a extração antes de criar o índice. Reexecute a célula de extração após mudar a configuração; a chave de cache mudará.

In [5]:
from collections import Counter

tamanhos = [r["tokens"] for r in registros]
print(f"Chunks: {len(registros)} | tokens mín/mediana/máx: "
      f"{min(tamanhos)}/{median(tamanhos):g}/{max(tamanhos)}")
print("Chunks com mais de uma página:", sum(len(r["paginas_pdf"]) > 1 for r in registros))
print("Chunks com tabela:", sum(any("table" in t.lower() for t in r["tipos_itens"])
                                for r in registros))
print("Páginas físicas cobertas:", min(p for r in registros for p in r["paginas_pdf"]),
      "a", max(p for r in registros for p in r["paginas_pdf"]))
cobertas = {p for r in registros for p in r["paginas_pdf"]}
faltantes = sorted(set(range(5, max(cobertas) + 1)) - cobertas)
if faltantes:
    raise ValueError(f"Páginas do corpo sem chunks: {faltantes}; revise a extração")

ocorrencias = Counter(ref for r in registros for ref in r["itens_docling"])
repetidos = {ref for ref, n in ocorrencias.items() if n > 1}
print("Itens Docling repartidos em múltiplos chunks:", len(repetidos))
for r in registros:
    if any(ref in repetidos for ref in r["itens_docling"]):
        print("  conferir:", r["id"], "páginas", r["paginas_pdf"],
              "títulos", r["titulos"][-2:], "tipos", r["tipos_itens"])

for secao in ("4.3.2", "7.4", "8.2"):
    candidatos = [r for r in registros if any(secao in t for t in r["titulos"])]
    print(f"\n=== Seção {secao}: {len(candidatos)} chunks ===")
    for r in candidatos[:3]:
        print(r["id"], "PDF", r["paginas_pdf"], "tokens", r["tokens"])
        print(r["texto"][:1300], "\n")
    if not candidatos:
        print("Título não reconhecido; revise a hierarquia extraída pelo Docling.")

Chunks: 96 | tokens mín/mediana/máx: 22/177/770
Chunks com mais de uma página: 31
Chunks com tabela: 46
Páginas físicas cobertas: 5 a 62
Itens Docling repartidos em múltiplos chunks: 0

=== Seção 4.3.2: 1 chunks ===
manual_0021 PDF [13, 14] tokens 246
4 Regras para carteiras e segmentações de clientes
4.3 Exemplos de uso de carteiras
4.3.2 Cenário 2 - Filtro adicional sobre os clientes com quando
Além de selecionar os clientes, é possível filtrar o resultado com a cláusula quando:
```
version 1.0 time "superv marcos": vendedores todostelevendas time "superv maria": vendedores todosdecampo carteira "televendas fixa com ruptura de campo": clientes ruptura("superv maria" para "superv marcos") de positivacao(90 dias) clientes carteirafixa("superv marcos") quando recencia de visita entre 10 dias e 90 dias
```
O filtro quando recencia de visita entre 10 dias e 90 dias na linha 13 garante que apenas clientes cuja última visita ocorreu entre 10 e 90 dias atrás serão incluídos. Filtros com quan

## Criar o índice vetorial do manual

O texto embutido é o chunk contextualizado, com a trilha de títulos. ID, páginas e arquivo ficam como metadados retornados na busca; não viram parte do vetor. O modelo é o mesmo `BAAI/bge-m3` usado no índice das explicações. Em T4: FP16 e lote 4; em CPU: FP32 e lote 2. O limite do modelo é arredondado acima do maior chunk e verificado antes da indexação, evitando truncamento silencioso.

In [6]:
from llama_index.core import Settings, StorageContext, VectorStoreIndex, load_index_from_storage
from llama_index.core.schema import MetadataMode, TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

modelo_tokenizer = AutoTokenizer.from_pretrained(MODELO_EMBEDDING)
maior = max(len(modelo_tokenizer.encode(r["texto"], truncation=False)) for r in registros)
if maior > MAX_MODELO_TOKENS:
    raise ValueError(f"Maior chunk tem {maior} tokens, acima de {MAX_MODELO_TOKENS}")
comprimento_modelo = max(256, ((maior + 127) // 128) * 128)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
lote = 4 if dispositivo == "cuda" else 2
modelo = HuggingFaceEmbedding(
    model_name=MODELO_EMBEDDING,
    device=dispositivo,
    embed_batch_size=lote,
    max_length=comprimento_modelo,
    model_kwargs={"torch_dtype": torch.float16} if dispositivo == "cuda" else {},
)
Settings.llm = None
Settings.embed_model = modelo
print("Embedding:", MODELO_EMBEDDING, "|", dispositivo, "| lote", lote,
      "| max_length", comprimento_modelo)

nos = []
for r in registros:
    metadados = {
        "fonte": REL_PDF.as_posix(),
        "chunk_id": r["id"],
        "paginas_pdf": r["paginas_pdf"],
        "titulos": r["titulos"],
    }
    no = TextNode(
        id_=r["id"], text=r["texto"], metadata=metadados,
        excluded_embed_metadata_keys=list(metadados),
        excluded_llm_metadata_keys=list(metadados),
    )
    qtd = len(modelo_tokenizer.encode(no.get_content(metadata_mode=MetadataMode.EMBED),
                                     truncation=False))
    if qtd > comprimento_modelo:
        raise ValueError(f"{r['id']} excede max_length; revise a divisão")
    nos.append(no)

MANIFESTO = PASTA_INDICE / "manifesto.json"
if MANIFESTO.is_file():
    indice_manual = load_index_from_storage(
        StorageContext.from_defaults(persist_dir=str(PASTA_INDICE)), embed_model=modelo
    )
    print("índice do manual carregado:", PASTA_INDICE)
else:
    indice_manual = VectorStoreIndex(nos, embed_model=modelo, show_progress=True)
    PASTA_INDICE.mkdir(parents=True, exist_ok=True)
    indice_manual.storage_context.persist(persist_dir=str(PASTA_INDICE))
    MANIFESTO.write_text(json.dumps({**CONFIG, "chunks": len(registros),
                                    "comprimento_modelo": comprimento_modelo,
                                    "dispositivo": dispositivo},
                                   ensure_ascii=False, indent=2), encoding="utf-8")
    print("índice do manual criado:", PASTA_INDICE)

`torch_dtype` is deprecated! Use `dtype` instead!


LLM is explicitly disabled. Using MockLLM.
Embedding: BAAI/bge-m3 | cuda | lote 4 | max_length 896


Generating embeddings:   0%|          | 0/96 [00:00<?, ?it/s]

índice do manual criado: /content/gobr_index_cache/manual/fd183f3b452fff50a249/indice


## Testar recuperação com fonte

A busca devolve seção e página **física do PDF**. O número impresso no rodapé do manual é quatro unidades menor nas páginas de conteúdo. Uma similaridade alta indica um trecho provável; confira o texto e a página antes de usar a resposta em um RAG.

In [7]:
def buscar_manual(pergunta: str, top_k: int = 5) -> list[dict]:
    if not pergunta.strip() or top_k < 1:
        raise ValueError("Informe pergunta não vazia e top_k positivo")
    hits = indice_manual.as_retriever(similarity_top_k=top_k).retrieve(pergunta)
    return [{
        "id": h.node.metadata["chunk_id"],
        "score": h.score,
        "fonte": h.node.metadata["fonte"],
        "paginas_pdf": h.node.metadata["paginas_pdf"],
        "titulos": h.node.metadata["titulos"],
        "texto": h.node.text,
    } for h in hits]

perguntas = [
    "Qual a diferença entre carteira exclusiva, carteira livre e segmentação?",
    "Como definir frequência de atendimento nos parâmetros de uma carteira?",
    "Como configurar um alerta de ruptura e escolher o canal de entrega?",
]
for pergunta in perguntas:
    print("\nPERGUNTA:", pergunta)
    for hit in buscar_manual(pergunta, top_k=3):
        print(hit["id"], "score", round(hit["score"], 4),
              "páginas PDF", hit["paginas_pdf"], "títulos", hit["titulos"][-2:])
        print(hit["texto"][:550].replace("\n", " "), "\n")


PERGUNTA: Qual a diferença entre carteira exclusiva, carteira livre e segmentação?
manual_0019 score 0.6639 páginas PDF [12, 13] títulos ['4 Regras para carteiras e segmentações de clientes', '4.2 Carteiras exclusivas vs. carteiras livres']
4 Regras para carteiras e segmentações de clientes 4.2 Carteiras exclusivas vs. carteiras livres Existe ainda um segundo tipo de carteira chamada carteira livre: | Tipo             | Comportamento                                                                                                                                                                                                                | |------------------|---------------------------------------------------------------------------------------------------------------------------- 

manual_0018 score 0.6384 páginas PDF [12] títulos ['4.1 O que são carteiras e segmentações?', '4.1.2 Segmentação']
4 Regras para carteiras e segmentações de clientes 4.1 O que são carteiras e segmentações?

## Como usar depois

`indice_manual` recupera trechos do manual. `buscar_manual(pergunta)` retorna texto e páginas. O índice persistido fica em `PASTA_INDICE`, separado de `gobr_index_cache/explicacoes`. Na entrega de RAG, recupere do manual para fundamentar sintaxe e restrições e do índice de explicações para encontrar regras anteriores. Esta etapa só indexa e demonstra buscas; não gera uma parametrização nem responde com um LLM.